# Komp. A — HydroBASINS Level 9 exploration

Visual comparison of HydroBASINS Level 9 (~100 km² basins) vs the
current Level 8 refined polygon (~700 km² basins). Run *before*
deciding whether to switch the refine-region default to L9.

Plots:
- All Level-9 basins intersecting the Inntal/Steinberge seed bbox,
  individually colored with their HYBAS_ID labels.
- Their majority-inside (`overlap_frac ≥ 0.5`) subset highlighted.
- The hauptkamm filter (north-of-Inn) applied as a second-level filter.
- The current L8 refined polygon (green outline) for direct comparison.

First run is slow (downloads `hybas_eu_lev09_v1c.zip`, ~51 MB).

In [ ]:
from pathlib import Path

import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import LineString, box

from alptherm_icon.regions.basins import (
    INN_LINE,
    basins_inside_bbox,
    fetch_hydrobasins,
    filter_north_of_inn,
)
from alptherm_icon.regions.polygon import load_region

REGION = "inntal_steinberge"
PROJECT_ROOT = Path.cwd().resolve().parents[0]
CONFIG_PATH = PROJECT_ROOT / "configs" / "regions" / f"{REGION}.geojson"

# Read the current refined polygon + its original seed bounds.
refined_geom, props = load_region(CONFIG_PATH, name=REGION)
seed_bounds = tuple(props.get("seed_bounds", refined_geom.bounds))
seed_bbox = box(*seed_bounds)
print(f"seed bbox:      {seed_bounds}")
print(f"current refined (L{props.get('source', 'lev08')[-2:]}): {props.get('n_basins')} basins")

In [ ]:
shp_path = fetch_hydrobasins(PROJECT_ROOT / "data" / "basins", region="eu", level=9)
basins_all = gpd.read_file(shp_path)
print(f"loaded {len(basins_all)} L9 basins from {shp_path.name}")

intersecting = basins_all[basins_all.intersects(seed_bbox)].copy()
majority = basins_inside_bbox(basins_all, seed_bbox, min_overlap_frac=0.5)
majority_north = filter_north_of_inn(majority)
print(f"  intersecting seed bbox:       {len(intersecting)}")
print(f"  + majority inside (>=50%):   {len(majority)}")
print(f"  + north-of-Inn hauptkamm:    {len(majority_north)}")
if "SUB_AREA" in intersecting.columns:
    print(
        f"  area distribution [km^2]: "
        f"min={intersecting.SUB_AREA.min():.1f} "
        f"median={intersecting.SUB_AREA.median():.1f} "
        f"max={intersecting.SUB_AREA.max():.1f}"
    )

In [ ]:
# Reproject everything to Web Mercator for the contextily basemap.
intersecting_3857 = intersecting.to_crs(3857)
majority_3857 = majority.to_crs(3857)
majority_north_3857 = majority_north.to_crs(3857)
refined_3857 = gpd.GeoDataFrame(geometry=[refined_geom], crs="EPSG:4326").to_crs(3857)
seed_3857 = gpd.GeoDataFrame(geometry=[seed_bbox], crs="EPSG:4326").to_crs(3857)

# Inn-line segment across the seed bbox.
a, b = INN_LINE
x0, _, x1, _ = seed_bounds
inn_line = LineString([(x0, a + b * x0), (x1, a + b * x1)])
inn_3857 = gpd.GeoDataFrame(geometry=[inn_line], crs="EPSG:4326").to_crs(3857)

cities = {
    "Innsbruck": (11.394, 47.265),
    "Schwaz": (11.708, 47.343),
    "Wörgl": (12.077, 47.485),
    "Kufstein": (12.171, 47.583),
}
cities_3857 = gpd.GeoDataFrame(
    {"name": list(cities.keys())},
    geometry=gpd.points_from_xy([v[0] for v in cities.values()], [v[1] for v in cities.values()]),
    crs="EPSG:4326",
).to_crs(3857)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))

# Layer 1: all intersecting L9 basins, each in a distinct tab20 color (light fill).
cmap = plt.get_cmap("tab20")
for i, (_, row) in enumerate(intersecting_3857.iterrows()):
    color = cmap(i % cmap.N)
    gpd.GeoDataFrame(geometry=[row.geometry], crs=intersecting_3857.crs).plot(
        ax=ax, facecolor=color, edgecolor=color, alpha=0.30, linewidth=0.4,
    )

# Layer 2: majority-inside-and-north-of-Inn basins — outlined heavier.
majority_north_3857.boundary.plot(ax=ax, color="#c0392b", linewidth=1.2)

# Layer 3: the current L8 refined polygon (green outline) for direct comparison.
refined_3857.boundary.plot(ax=ax, color="#1e8449", linewidth=2.0)

# Seed bbox + Inn line.
seed_3857.boundary.plot(ax=ax, color="#7f8c8d", linewidth=1.2, linestyle="--")
inn_3857.plot(ax=ax, color="#2471a3", linewidth=1.5, linestyle="--")

# HYBAS_ID labels at each basin's representative point (only for intersecting basins).
for _, row in intersecting_3857.iterrows():
    pt = row.geometry.representative_point()
    ax.annotate(
        str(int(row["HYBAS_ID"]))[-4:],  # last 4 digits for legibility
        xy=(pt.x, pt.y),
        fontsize=7, color="black", ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.6),
    )

# Cities.
for _, row in cities_3857.iterrows():
    ax.plot(row.geometry.x, row.geometry.y, marker="o", color="black", markersize=4)
    ax.annotate(
        row["name"], xy=(row.geometry.x, row.geometry.y),
        xytext=(5, 5), textcoords="offset points", fontsize=9, color="black",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75),
    )

# Frame on the seed bbox with a small margin.
minx, miny, maxx, maxy = seed_3857.total_bounds
margin = 0.06 * max(maxx - minx, maxy - miny)
ax.set_xlim(minx - margin, maxx + margin)
ax.set_ylim(miny - margin, maxy + margin)
cx.add_basemap(ax, source=cx.providers.OpenTopoMap, crs=refined_3857.crs, zoom=10)

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

ax.legend(
    handles=[
        Patch(facecolor="gray", edgecolor="gray", alpha=0.30,
              label=f"L9 basins intersecting bbox ({len(intersecting)})"),
        Line2D([0], [0], color="#c0392b", linewidth=1.2,
               label=f"L9 majority-inside + N-of-Inn ({len(majority_north)})"),
        Line2D([0], [0], color="#1e8449", linewidth=2.0,
               label=f"current L8 refined ({props.get('n_basins')})"),
        Line2D([0], [0], color="#7f8c8d", linewidth=1.2, linestyle="--",
               label="seed bbox"),
        Line2D([0], [0], color="#2471a3", linewidth=1.5, linestyle="--",
               label="Inn-line"),
    ],
    loc="lower right", framealpha=0.9, fontsize=8,
)
ax.set_title("Komp. A — L9 basin exploration (Inntal/Steinberge)")
ax.set_xticks([])
ax.set_yticks([])
fig.tight_layout()

out = PROJECT_ROOT / "data" / "regions" / f"{REGION}_basins_lev09.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"saved {out.relative_to(PROJECT_ROOT)}")

## Decision questions

- Are individual L9 basins fine-grained enough to follow actual
  sub-catchments visible in the basemap (Karwendel side valleys,
  Brandenberg tributaries)?
- Does the L9 majority-inside-and-N-of-Inn selection (red outlines)
  cover roughly the same area as the L8 refined polygon (green
  outline), or does it leak into the foreland in a way L8 didn't?
- How many L9 basins survive selection? A handful (~10-20) is
  workable; ~100 starts to dilute the "one representative profile
  per region" assumption of Komp. C.

If L9 looks better, switch the default via:

```bash
python -m alptherm_icon.regions refine-region inntal_steinberge --level 9
```